# Notebook 04c: 2nd-Stage Adaptive Refinement of Post-Hoc Model (`04c_cmf_adaptive.ipynb`)

### Objectives & Workflow:
1. **2nd-Stage Adaptive Refinement of 04b**: Starting directly from the post-hoc aligned model of Notebook 04b (`cmf_04b_*.pt`), preserve the 04b-aligned classifier head (freeze head parameters) and execute a gentle 2nd-stage adaptive encoder unlearning refinement across all 5 methods:
   - **Random Label**
   - **NegGrad+**
   - **SalUn**
   - **SCRUB**
   - **TarUn / UNSIR**
2. **Protocol Definition**: **04b Model -> Adaptive Encoder Refinement (low-LR) -> Final Model** (no head reset or destructive overwriting).
3. **Consolidated Multi-Tier Benchmark**: Compute Output Acc, Linear Probe Acc, NCC Acc, and Privacy / MIA metrics on $D_V$, $D_R$, $D_F$, and $D_T$.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    compute_output_accuracy,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    print_metrics_summary_table,
    compute_class_mean_geometry,
    compare_class_mean_geometry,
    load_oracle_geometry,
    print_geometry_summary_table,
    MODEL_CONFIGS
)

from unlearning_methods import (
    unlearn_random_label,
    unlearn_neggrad_plus,
    unlearn_salun,
    unlearn_scrub,
    unlearn_tarun_unsir
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. CMF Head Reconstruction Utility

In [ ]:
# CMF head reconstruction (Gao et al., arXiv:2604.08271v1, Algorithm 1) lives in experiment_utils:
#   mu_k = class-mean feature, mu_bar = (1/K) sum_k mu_k, w_k = (mu_k - mu_bar) / ||mu_k - mu_bar||,
#   W_CMF = [w_1..w_K] with zero bias, frozen.
from experiment_utils import reconstruct_cmf_head


## 2. Iterative & Adaptive CMF Refinement Loop Across All 5 Methods

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # from MODEL_CONFIGS[DATASET_NAME]['model_seeds']
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

METHODS = ["random_label", "neggrad_plus", "salun", "scrub", "tarun_unsir"]

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
from data_loader import load_raw_dataset as _load_raw_dataset
raw_train, raw_test = _load_raw_dataset(DATASET_NAME, root="./data")
if raw_test is None:
    raise ValueError(f"Dataset '{DATASET_NAME}' has no separate test split via load_raw_dataset; handle UTKFace via build_unlearning_splits instead.")

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/plots", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")

with open(split_file, "r") as f:
    split_info = json.load(f)

# DataLoaders
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_f_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_f_loader = DataLoader(train_f_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

all_04c_results = []

for seed in MODEL_SEEDS:
    for method_name in METHODS:
        # Load 04b checkpoint
        kaggle_ckpt_paths = [
            f"/kaggle/input/datasets/kiethe/experiment-nb4a-c10/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/experiment-nb4a-c10/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/cifar10-experimentnb1/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            # Fallback
            f"/kaggle/input/datasets/kiethe/experiment-nb4a-c10/cmf_04b_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/cmf_04b_{MODEL_NAME}_seed_{seed}.pt",
        ]
        local_ckpt_04b = f"./artifacts/checkpoints/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        fallback_local = f"./artifacts/checkpoints/cmf_04b_{MODEL_NAME}_seed_{seed}.pt"
        
        ckpt_04b = next((p for p in kaggle_ckpt_paths if os.path.exists(p)), local_ckpt_04b if os.path.exists(local_ckpt_04b) else fallback_local)
        
        if not os.path.exists(ckpt_04b):
            raise FileNotFoundError(f"04b checkpoint missing for method {method_name} seed {seed} at {ckpt_04b}. Run Notebook 04b first.")
        print(f"Using 04b ckpt: {ckpt_04b}")

        print(f"\n==================================================")
        print(f"--- Running 04c: Adaptive CMF | Method: {method_name} | Model: {MODEL_NAME} | Seed: {seed} ---")
        print(f"==================================================")

        # Load 04b model weights
        model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(ckpt_04b, map_location=device))

        # 1. Freeze 04b-aligned classifier head to preserve decision boundaries during encoder refinement
        for p in model.head.parameters():
            p.requires_grad = False

        start_time = time.time()

        # 2. Run 2nd-stage gentle adaptive refinement on encoder
        if method_name == "random_label":
            unlearned_model = unlearn_random_label(model, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, lr=0.013, epochs=10, device=device)
        elif method_name == "neggrad_plus":
            unlearned_model = unlearn_neggrad_plus(model, train_r_loader, train_f_loader, alpha=0.95, lr=0.01, epochs=10, device=device)
        elif method_name == "salun":
            unlearned_model = unlearn_salun(model, train_r_loader, train_f_loader, num_classes=NUM_CLASSES, threshold=0.5, lr=0.013, epochs=10, device=device)
        elif method_name == "scrub":
            unlearned_model = unlearn_scrub(model, train_r_loader, train_f_loader, epochs=3, msteps=2, lr=5e-4, device=device)
        elif method_name == "tarun_unsir":
            unlearned_model = unlearn_tarun_unsir(model, train_r_loader, forget_loader=train_f_loader, num_classes=NUM_CLASSES, noise_lr=0.1, noise_epochs=5, noise_steps=8, impair_lr=0.02, repair_lr=0.01, device=device)

        runtime = time.time() - start_time

        # Save unlearned adaptive checkpoint
        save_ckpt = f"./artifacts/checkpoints/cmf_04c_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        torch.save(unlearned_model.state_dict(), save_ckpt)

        # Evaluate 3-tier metrics
        # Validation metric: output accuracy on the held-out D_V split.
        val_metrics = {"output_val": compute_output_accuracy(unlearned_model, val_loader, device=device)}
        test_metrics = evaluate_three_tier_metrics(
            model=unlearned_model,
            probe_train_loader=retain_eval_loader,
            retain_eval_loader=retain_eval_loader,
            forget_eval_loader=forget_eval_loader,
            num_classes=NUM_CLASSES,
            seed=seed,
            device=device,
            test_eval_loader=test_loader,
            nonmember_loader=val_loader
        )

        # Class-mean geometry: global_mean -> class_mean directions, compared with the Oracle
        # (02_oracle_retrain, same seed) after orthogonal Procrustes alignment of feature bases.
        geom = compute_class_mean_geometry(
            unlearned_model, retain_eval_loader, forget_eval_loader, num_classes=NUM_CLASSES, device=device
        )
        geometry_vs_oracle = compare_class_mean_geometry(geom, load_oracle_geometry(MODEL_NAME, seed))

        res = {
            "variant": "04c_cmf_adaptive",
            "method": method_name,
            "model_name": MODEL_NAME,
            "seed": seed,
            "split_seed": SPLIT_SEED,
            "runtime_sec": runtime,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics,
            "geometry_vs_oracle": geometry_vs_oracle
        }
        all_04c_results.append(res)
        res_path = f"./artifacts/metrics/cmf_04c_{method_name}_{MODEL_NAME}_seed_{seed}.json"
        with open(res_path, "w") as f:
            json.dump(res, f, indent=2)

        print(f"[{method_name}] Geometry vs Oracle | Retain cos: {geometry_vs_oracle['retain']['cos_mean']:.4f} "
              f"| Retain norm ratio: {geometry_vs_oracle['retain']['norm_ratio_mean']:.4f} "
              f"| Forget cos: {geometry_vs_oracle['forget']['cos_mean']:.4f} "
              f"| Forget norm ratio: {geometry_vs_oracle['forget']['norm_ratio_mean']:.4f}")
        print(f"[{method_name}] 04c Retain (D_R) Output Acc: {test_metrics['output_retain']:.4f} | Forget (D_F) Output Acc: {test_metrics['output_forget']:.4f} | Test (D_T) Acc: {test_metrics.get('output_test', 0.0):.4f}")
        print(f"[{method_name}] 04c Retain (D_R) LP Acc:     {test_metrics['lp_retain']:.4f} | Forget (D_F) LP Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
        print(f"[{method_name}] 04c Retain (D_R) NCC Acc:    {test_metrics['ncc_retain']:.4f} | Forget (D_F) NCC Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
        print(f"[{method_name}] Saved 04c checkpoint & metrics to {res_path}")

        # Feature space & decision boundary visualization for first seed
        if seed == MODEL_SEEDS[0]:
            vis_save = f"./artifacts/plots/features_04c_cmf_{method_name}_{MODEL_NAME}_seed_{seed}.png"
            visualize_feature_space_and_boundaries(
                model=unlearned_model,
                retain_loader=retain_eval_loader,
                forget_loader=forget_eval_loader,
                num_classes=NUM_CLASSES,
                title=f"Feature Space: 04c CMF Adaptive ({method_name}, {MODEL_NAME})",
                device=device,
                save_path=vis_save
            )

# ============================================================
# Final Consolidated Benchmark Metrics Summary Table
# ============================================================
summary_df = print_metrics_summary_table(
    results=all_04c_results,
    title="04c CMF Adaptive Representation Refinement Metrics Summary",
    save_csv_path=f"./artifacts/metrics/summary_04c_cmf_{MODEL_NAME}.csv"
)

# ============================================================
# Class-Mean Direction Geometry vs Oracle (per method)
# ============================================================
geometry_df = print_geometry_summary_table(
    results=all_04c_results,
    title="04c CMF Adaptive: Class-Mean Direction Geometry vs Oracle (D_R)",
    save_csv_path=f"./artifacts/metrics/summary_geometry_04c_cmf_{MODEL_NAME}.csv"
)